In [1]:
import numpy as np


class SVD:
    def __init__(self):
        self.U = None
        self.S = None
        self.Vt = None

    def fit(self, A):
        """
        Compute SVD from scratch:
        A = U Σ V^T
        """

        m, n = A.shape

        # Step 1: Compute A^T A
        AtA = np.dot(A.T, A)

        # Step 2: Eigen decomposition of A^T A
        eigenvalues, V = np.linalg.eig(AtA)

        # Step 3: Sort eigenvalues in descending order
        idx = np.argsort(eigenvalues)[::-1]

        eigenvalues = eigenvalues[idx]
        V = V[:, idx]

        # Step 4: Singular values
        singular_values = np.sqrt(
            np.maximum(eigenvalues, 0)
        )

        # Step 5: Compute U
        U = []

        for i in range(len(singular_values)):
            sigma = singular_values[i]

            if sigma > 1e-10:
                u = np.dot(A, V[:, i]) / sigma
                U.append(u)

        U = np.column_stack(U)

        # Step 6: Create Sigma matrix
        Sigma = np.zeros((m, n))

        for i in range(min(m, n)):
            Sigma[i, i] = singular_values[i]

        self.U = U
        self.S = Sigma
        self.Vt = V.T

        return self

    def transform(self, k):
        """
        Return rank-k approximation
        """

        U_k = self.U[:, :k]
        S_k = self.S[:k, :k]
        Vt_k = self.Vt[:k, :]

        return U_k @ S_k @ Vt_k

    def reconstruct(self):
        """
        Full reconstruction
        """

        return self.U @ self.S[:self.U.shape[1], :] @ self.Vt

    def explained_variance(self):
        """
        Percentage contribution of singular values
        """

        singular_values = np.diag(self.S)

        variance = singular_values**2

        return variance / np.sum(variance)

In [2]:
# Example


if __name__ == "__main__":

    A = np.array([
        [1, 2],
        [3, 4],
        [5, 6]
    ], dtype=float)

    svd = SVD()
    svd.fit(A)

    print("\nMatrix A:")
    print(A)

    print("\nU:")
    print(svd.U)

    print("\nSigma:")
    print(svd.S)

    print("\nV^T:")
    print(svd.Vt)

    reconstructed = svd.reconstruct()

    print("\nReconstructed Matrix:")
    print(reconstructed)

    print("\nRank-1 Approximation:")
    print(svd.transform(k=1))

    print("\nExplained Variance:")
    print(svd.explained_variance())


Matrix A:
[[1. 2.]
 [3. 4.]
 [5. 6.]]

U:
[[-0.2298477   0.88346102]
 [-0.52474482  0.24078249]
 [-0.81964194 -0.40189603]]

Sigma:
[[9.52551809 0.        ]
 [0.         0.51430058]
 [0.         0.        ]]

V^T:
[[-0.61962948 -0.78489445]
 [-0.78489445  0.61962948]]

Reconstructed Matrix:
[[1. 2.]
 [3. 4.]
 [5. 6.]]

Rank-1 Approximation:
[[1.35662819 1.71846235]
 [3.09719707 3.92326845]
 [4.83776596 6.12807454]]

Explained Variance:
[0.99709335 0.00290665]
